In [46]:
from sympl import (NetCDFMonitor, AdamsBashforth)
import xarray as xr
import numpy as np
from datetime import timedelta
import matplotlib.pyplot as plt

from climt import (
    EmanuelConvection, RRTMGShortwave, RRTMGLongwave, SlabSurface,
    DryConvectiveAdjustment, SimplePhysics, get_default_state
)

In [47]:
from sympl import (AdamsBashforth, NetCDFMonitor)
import climt
from datetime import timedelta
# Define model timestep in minutes
model_timestep = timedelta(minutes=5)
# Create components
radiation = climt.RRTMGLongwave()
convection = climt.EmanuelConvection()
boundary_layer = climt.SimplePhysics()

# Create model state
model_state = climt.get_default_state([radiation, convection, boundary_layer])
# Create integrator
time_stepper = AdamsBashforth([radiation, convection])
# Create monitor
monitor = NetCDFMonitor('/home/wang.shuoc/climt/radiative_convective.nc')

# step model forward
for step in range(200):
    bl_diagnostics, bl_new_state = boundary_layer(model_state, model_timestep)
    model_state.update(bl_diagnostics)
    model_state.update(bl_new_state)
    diagnostics, new_state = time_stepper(model_state, model_timestep)
    model_state.update(diagnostics)
    monitor.store(model_state)
    model_state.update(new_state)
    model_state['time'] += model_timestep

monitor.write()

/projects/sds-lab/Shuochen/miniconda3/envs/ai/lib/python3.9/site-packages/sympl/_core/tendencystepper.py:147: UserWarning: Using an ImplicitTendencyComponent in sympl TendencyStepper objects may lead to scientifically invalid results. Make sure the component follows the same numerical assumptions as the TendencyStepper used.
  warnings.warn(


In [51]:
state = xr.open_dataset('radiative_convective.nc')
state

<xarray.Dataset>
Dimensions:                                                            (
                                                                        time: 200,
                                                                        interface_levels: 29,
                                                                        lat: 1,
                                                                        lon: 1,
                                                                        mid_levels: 28,
                                                                        ice_interface_levels: 30,
                                                                        num_longwave_bands: 16)
Coordinates:
  * time                                                               (time) datetime64[ns] ...
Dimensions without coordinates: interface_levels, lat, lon, mid_levels,
                                ice_interface_levels, num_longwave_bands
Data variables: (12/48)
    atmosphere_hybrid_sigma_pressure_a_coordinate_on_interface_levels  (time, interface_levels) float64 ...
    atmosphere_hybrid_sigma_pressure_b_coordinate_on_interface_levels  (time, interface_levels) float64 ...
    surface_air_pressure                                               (time, lat, lon) float64 ...
    air_pressure                                                       (time, mid_levels, lat, lon) float64 ...
    air_pressure_on_interface_levels                                   (time, interface_levels, lat, lon) float64 ...
    longitude                                                          (time, lat, lon) float64 ...
    ...                                                                 ...
    convective_precipitation_rate                                      (time, lat, lon) float64 ...
    convective_downdraft_velocity_scale                                (time, lat, lon) float64 ...
    convective_downdraft_temperature_scale                             (time, lat, lon) float64 ...
    convective_downdraft_specific_humidity_scale                       (time, lat, lon) float64 ...
    atmosphere_convective_available_potential_energy                   (time, lat, lon) float64 ...
    air_temperature_tendency_from_convection                           (time, lat, lon, mid_levels) float64 ...

In [39]:
state['air_temperature_tendency_from_longwave']

<xarray.DataArray (mid_levels: 28, lat: 1, lon: 1)>
array([[[  1.47124013]],

       [[  0.65961955]],

       [[  0.11533648]],

       [[ -0.118515  ]],

       [[ -0.22563343]],

       [[ -0.28516093]],

       [[ -0.32517453]],

       [[ -0.35605478]],

       [[ -0.38225291]],

       [[ -0.40565541]],

...

       [[ -0.69873188]],

       [[ -0.79406384]],

       [[ -0.95092903]],

       [[ -1.23477145]],

       [[ -1.67297183]],

       [[ -2.3229303 ]],

       [[ -3.51394001]],

       [[ -4.80016689]],

       [[ -6.23661925]],

       [[-11.36012646]]])
Dimensions without coordinates: mid_levels, lat, lon
Attributes:
    units:    degK day^-1